# Type 0 intervention: evaluation, LAION and COCO vocabulary

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import torch

from utils import obj_enc_aggregation
from intervention_image_level import (
    build_laion_lexicon, load_intervention_testset, load_gt_categories, load_category_id_by_name,
    load_checkpoint, load_variants, load_boundary_type, apply_intervention,
    balanced_accuracy, positive_variant_accuracy, nb_accuracy,
)
from intervention_image_level_coco import build_trivial_lexicon

ENC = "../encodings/cocologic"
OBJ = "obj_7_crop_min_0.01_max_0.85_score_0.2_iou_0.5_model_mask-rcnn_0.25"
TRAIN_JSON = "../data/cocologic/cocologic_train.json"
TEST_JSON = "../data/cocologic/cocologic_test.json"
K = 7
AGGREGATIONS = ["sum", "max"]

VOCABS = {
    "laion": {
        "lexicon": build_laion_lexicon(),
        "base": f"{ENC}/clip/splice_embeddings_0.25.h5",
        "obj": f"{ENC}/obj_encodings/clip/{OBJ}.h5",
        "ckpt_tag": "",
    },
    "coco_categories": {
        "lexicon": build_trivial_lexicon("../datasets/coco/annotations/instances_val2017.json"),
        "base": f"{ENC}/clip/splice_embeddings_0.25_coco_categories.h5",
        "obj": f"{ENC}/obj_encodings/clip/{OBJ}_coco_categories.h5",
        "ckpt_tag": "_coco_categories",
    },
}
for vocab, v in VOCABS.items():
    v["targets"] = {
        agg: torch.load(f"../data/cocologic/percentile_targets_full_train_{agg}_{vocab}.pt")
        for agg in AGGREGATIONS
    }

category_id_by_name = load_category_id_by_name(TRAIN_JSON)

RULE_NAMES = ["Signal and Ride", "Double Serving", "Herd Alone", "Dog or Car", "Three of a Kind",
              "Car Majority", "Empty Seat", "Single Mode Traffic", "Personal Transport", "Surf Trip"]

PERSON_RULES = ["Herd Alone", "Empty Seat", "Personal Transport", "Surf Trip"]

In [ ]:
# test set 
test_ids = None
for vocab, v in VOCABS.items():
    emb, obj, img_ids, labels = load_intervention_testset(v["base"], v["obj"], K)
    v["test"] = {"emb": emb, "obj": obj[:, :K]}
    v["raw_min_test"] = min(emb.min().item(), obj.min().item())
    if test_ids is None:
        test_ids, test_labels = img_ids, labels.float()
    assert torch.equal(img_ids, test_ids)

gt_test = load_gt_categories(TEST_JSON, test_ids)
variants = load_variants(TEST_JSON, test_ids)
boundary_type = load_boundary_type(TEST_JSON, test_ids)
print(len(test_ids), "test images")

## Vocabulary check


In [ ]:
rows = []
for vocab, v in VOCABS.items():
    emb, obj, train_ids, _ = load_intervention_testset(v["base"], v["obj"], K, split="train")
    v["raw_min_train"] = min(emb.min().item(), obj.min().item())
    agg_train = obj_enc_aggregation("sum", obj[:, :K], emb, only_objects=False)
    del emb, obj
    gt_train = load_gt_categories(TRAIN_JSON, train_ids)
    for cat, idxs in v["lexicon"].items():
        if not idxs:
            continue
        present = gt_train[:, category_id_by_name[cat]] > 0
        act = agg_train[:, idxs].mean(dim=1)
        rows.append({
            "vocab": vocab, "category": cat, "n_with": int(present.sum()),
            "mean_with": act[present].mean().item(), "mean_without": act[~present].mean().item(),
        })
    del agg_train

plausibility = pd.DataFrame(rows)
plausibility["ratio"] = plausibility["mean_with"] / plausibility["mean_without"]
not_clearly_higher = plausibility[~(plausibility["ratio"] >= 2)]
print(not_clearly_higher.sort_values(["vocab", "ratio"]).round(4).to_string(index=False))

## Main results


In [ ]:
def per_rule(logits):
    return {
        "B-Acc": balanced_accuracy(logits, test_labels),
        "PosVar-Acc": positive_variant_accuracy(logits, variants),
        "NB-Acc": nb_accuracy(logits, boundary_type),
    }

def rule_mean(logits):
    return {k: 100 * torch.nanmean(v).item() for k, v in per_rule(logits).items()}

def predict(model, x):
    with torch.no_grad():
        return model(x)

def count_overwrites(base, targets, kind):

    counts = {"all": [0, 0], "person": [0, 0]}
    for cat, t in targets.items():
        present = gt_test[:, category_id_by_name[cat]] > 0
        mask = present if kind == "boost" else ~present
        idxs = list(t[kind].keys())
        values = base[mask][:, idxs]
        target = torch.tensor([t[kind][i] for i in idxs], dtype=values.dtype)
        hits = (target < values) if kind == "boost" else (target > values)
        for key in ["all"] + (["person"] if cat == "person" else []):
            counts[key][0] += int(hits.sum())
            counts[key][1] += values.numel()
    return counts

results, changes, overwrites, outputs = [], [], [], {}
for vocab, v in VOCABS.items():
    for agg in AGGREGATIONS:
        model = load_checkpoint(f"../checkpoints/cocologic/{agg}_super_multilabel_seed42{v['ckpt_tag']}.pt")
        targets = v["targets"][agg]
        base = obj_enc_aggregation(agg, v["test"]["obj"], v["test"]["emb"], only_objects=False)
        boost = apply_intervention(base, gt_test, targets, category_id_by_name, suppress=False)
        full = apply_intervention(base, gt_test, targets, category_id_by_name, suppress=True)
        logits = {"none": predict(model, base), "boost": predict(model, boost), "boost+suppress": predict(model, full)}
        outputs[(vocab, agg)] = {"model": model, "base": base, "logits": logits}

        for cond, lg in logits.items():
            results.append({"vocab": vocab, "agg": agg, "intervention": cond, **rule_mean(lg)})

        pred_boost = torch.sigmoid(logits["boost"]) > 0.5
        pred_full = torch.sigmoid(logits["boost+suppress"]) > 0.5
        changes.append({
            "vocab": vocab, "agg": agg,
            "vector changed by suppress (%)": 100 * (boost != full).any(dim=1).float().mean().item(),
            "any rule prediction changed (%)": 100 * (pred_boost != pred_full).any(dim=1).float().mean().item(),
        })

        b = count_overwrites(base, targets, "boost")
        s = count_overwrites(base, targets, "suppress")
        overwrites.append({
            "vocab": vocab, "agg": agg,
            "boost lowers (%)": 100 * b["all"][0] / b["all"][1], "boost pairs": b["all"][1],
            "boost lowers person (%)": 100 * b["person"][0] / max(b["person"][1], 1), "person pairs": b["person"][1],
            "suppress raises (%)": 100 * s["all"][0] / s["all"][1], "suppress pairs": s["all"][1],
            "min aggregated": base.min().item(),
            "min raw test": v["raw_min_test"], "min raw train": v["raw_min_train"],
        })

print("Table 2: mean over rules (%)")
print(pd.DataFrame(results).round(2).to_string(index=False))
print()
print("Table 3: effect of suppress vs. boost only")
print(pd.DataFrame(changes).round(2).to_string(index=False))
print()
print("Table 4: overwrites that go against the intended direction")
print(pd.DataFrame(overwrites).round(3).to_string(index=False))

## Person


In [ ]:
person_id = category_id_by_name["person"]
has_person = gt_test[:, person_id] > 0
rule_idxs = [RULE_NAMES.index(r) for r in PERSON_RULES]

counts, person_rows = [], []
for vocab, v in VOCABS.items():
    person_dims = v["lexicon"]["person"]
    for agg in AGGREGATIONS:
        out = outputs[(vocab, agg)]
        base = out["base"]
        active = (base[:, person_dims] > 0).any(dim=1)
        counts.append({
            "vocab": vocab, "agg": agg, "test images with person": int(has_person.sum()),
            "person > 0 before (of those)": int((active & has_person).sum()),
            "person > 0 before (%)": 100 * (active & has_person).sum().item() / has_person.sum().item(),
        })

        targets = {c: t for c, t in v["targets"][agg].items() if c != "person"}
        no_person = apply_intervention(base, gt_test, targets, category_id_by_name, suppress=True)
        conditions = {
            "none": out["logits"]["none"],
            "boost+suppress": out["logits"]["boost+suppress"],
            "without person": predict(out["model"], no_person),
        }
        metrics = {cond: per_rule(lg) for cond, lg in conditions.items()}
        for r in rule_idxs:
            row = {"vocab": vocab, "agg": agg, "rule": RULE_NAMES[r]}
            for metric in ["B-Acc", "PosVar-Acc", "NB-Acc"]:
                for cond in conditions:
                    row[f"{metric} {cond}"] = 100 * metrics[cond][metric][r].item()
            person_rows.append(row)

print(pd.DataFrame(counts).round(2).to_string(index=False))
print()
print("Per-rule metrics (%) for the rules with person")
print(pd.DataFrame(person_rows).round(2).to_string(index=False))

In [ ]:
# boost value of person 
for vocab, v in VOCABS.items():
    for agg in AGGREGATIONS:
        boost = list(v["targets"][agg]["person"]["boost"].values())
        print(f"{vocab} {agg}: {len(boost)} person dimensions, all boost targets are 0: {all(b == 0 for b in boost)}")

## Linear weights (COCO)



In [ ]:
# rule-relevant categories per rule, COCOLogic-V2 Table 1 ("Either Dog or Car" is named "Dog or Car" here)
RULE_RELEVANT = {
    "Signal and Ride": ["traffic light", "bicycle", "bus", "train"],
    "Double Serving": ["bottle", "cup", "pizza"],
    "Herd Alone": ["cow", "elephant", "sheep", "person"],
    "Dog or Car": ["dog", "car"],
    "Three of a Kind": ["bowl", "cup"],
    "Car Majority": ["car", "truck"],
    "Empty Seat": ["couch", "chair", "person"],
    "Single Mode Traffic": ["bicycle", "motorcycle", "car", "bus"],
    "Personal Transport": ["person", "bicycle", "car"],
    "Surf Trip": ["person", "surfboard"],
}
RELEVANT_19 = list(dict.fromkeys(c for cats in RULE_RELEVANT.values() for c in cats))
coco_lexicon = VOCABS["coco_categories"]["lexicon"]

for agg in AGGREGATIONS:
    layer = outputs[("coco_categories", agg)]["model"].classifier
    weight, bias = layer.weight.detach(), layer.bias.detach()
    assert weight.shape == (len(RULE_NAMES), len(coco_lexicon))
    relevant_rows, other_rows = [], []
    for r, rule in enumerate(RULE_NAMES):
        row = {"rule": rule, "bias": bias[r].item()}
        for cat in RELEVANT_19:
            row[cat] = weight[r, coco_lexicon[cat][0]].item() if cat in RULE_RELEVANT[rule] else float("nan")
        relevant_rows.append(row)
        others = [(cat, weight[r, idxs[0]].item()) for cat, idxs in coco_lexicon.items()
                  if cat not in RULE_RELEVANT[rule]]
        top = sorted(others, key=lambda x: -abs(x[1]))[:5]
        other_rows.append({"rule": rule, **{f"#{i + 1}": f"{cat} {w:+.3f}" for i, (cat, w) in enumerate(top)}})
    print(f"[{agg}] weights on rule-relevant categories (empty = not relevant for the rule) and bias")
    print(pd.DataFrame(relevant_rows).set_index("rule").round(3).to_string(na_rep=""))
    print()
    print(f"[{agg}] 5 largest |weights| on non-relevant categories")
    print(pd.DataFrame(other_rows).to_string(index=False))
    print()

 positives and near-boundary negatives 

In [ ]:
import numpy as np

coco_cat_ids = [category_id_by_name[c] for c in VOCABS["coco_categories"]["lexicon"]]
present = gt_test > 0
patterns = present[:, coco_cat_ids].int()
n_patterns = len(torch.unique(patterns, dim=0))

full_vectors, rows = {}, []
for vocab, v in VOCABS.items():
    for agg in AGGREGATIONS:
        base = outputs[(vocab, agg)]["base"]
        targets = v["targets"][agg]
        full = apply_intervention(base, gt_test, targets, category_id_by_name, suppress=True)
        full_vectors[(vocab, agg)] = full
        target_dims = {i for t in targets.values() for i in list(t["boost"]) + list(t["suppress"])}
        rows.append({
            "vocab": vocab, "agg": agg, "dims": base.shape[1], "overwritten dims": len(target_dims),
            "never overwritten (%)": 100 * (1 - len(target_dims) / base.shape[1]),
            "values equal to before (%)": 100 * (full == base).float().mean().item(),
            "distinct vectors": len(torch.unique(full, dim=0)),
            "distinct GT patterns": n_patterns,
            "distinct (pattern, vector)": len(torch.unique(torch.cat([patterns.float(), full], dim=1), dim=0)),
        })
print(pd.DataFrame(rows).round(2).to_string(index=False))

def share_like_positive(keys, nb_mask, pos_mask):
    pos_keys = {k.numpy().tobytes() for k in keys[pos_mask]}
    hits = [k.numpy().tobytes() in pos_keys for k in keys[nb_mask]]
    return f"{100 * np.mean(hits):.2f} % ({sum(hits)})"

rows = []
for rule in ["Car Majority", "Surf Trip", "Three of a Kind"]:
    r = RULE_NAMES.index(rule)
    nb_mask = boundary_type[:, r] != 0
    pos_mask = test_labels[:, r] == 1
    relevant_ids = [category_id_by_name[c] for c in RULE_RELEVANT[rule]]
    row = {
        "rule": rule, "NB images": int(nb_mask.sum()), "positives": int(pos_mask.sum()),
        "same 80-category pattern": share_like_positive(patterns, nb_mask, pos_mask),
        "same pattern on relevant categories": share_like_positive(present[:, relevant_ids].int(), nb_mask, pos_mask),
    }
    for (vocab, agg), full in full_vectors.items():
        row[f"same vector {vocab} {agg}"] = share_like_positive(full, nb_mask, pos_mask)
    rows.append(row)
print()
print(pd.DataFrame(rows).T.to_string(header=False))

## Per-rule tables 

In [ ]:
SETTINGS = [("laion", "sum"), ("laion", "max"), ("coco_categories", "sum"), ("coco_categories", "max")]
SETTING_NAMES = {"laion": "LAION", "coco_categories": "COCO"}
METRIC_NAMES = ["B-Acc", "PosVar-Acc", "NB-Acc"]
table2 = pd.DataFrame(results).set_index(["vocab", "agg", "intervention"])

for vocab, agg in SETTINGS:
    logits = outputs[(vocab, agg)]["logits"]
    before, after = per_rule(logits["none"]), per_rule(logits["boost+suppress"])
    lines = [f"**{SETTING_NAMES[vocab]} {agg}**", "", "| Rule | " + " | ".join(METRIC_NAMES) + " |", "|---|---|---|---|"]
    for r, rule in enumerate(RULE_NAMES):
        cells = [f"{100 * before[m][r].item():.2f} -> {100 * after[m][r].item():.2f}" for m in METRIC_NAMES]
        lines.append(f"| {rule} | " + " | ".join(cells) + " |")
    means = []
    for m in METRIC_NAMES:
        b, a = 100 * torch.nanmean(before[m]).item(), 100 * torch.nanmean(after[m]).item()
        # mean rows must match the overall results (Table 2)
        assert np.isclose(b, table2.loc[(vocab, agg, "none"), m]), (vocab, agg, m, b)
        assert np.isclose(a, table2.loc[(vocab, agg, "boost+suppress"), m]), (vocab, agg, m, a)
        means.append(f"{b:.2f} -> {a:.2f}")
    lines.append("| **Mean** | " + " | ".join(means) + " |")
    print("\n".join(lines))
    print()
print("mean rows match Table 2")

## Per-rule table 

In [ ]:
THESIS_ORDER = ["Signal and Ride", "Empty Seat", "Double Serving", "Dog or Car", "Single Mode Traffic",
                "Personal Transport", "Herd Alone", "Three of a Kind", "Car Majority", "Surf Trip"]
logits = outputs[("coco_categories", "sum")]["logits"]
before, after = per_rule(logits["none"]), per_rule(logits["boost+suppress"])
rows = []
for rule in THESIS_ORDER:
    r = RULE_NAMES.index(rule)
    row = {"rule": "Either Dog or Car" if rule == "Dog or Car" else rule}
    for m in METRIC_NAMES:
        row[f"{m} base"] = 100 * before[m][r].item()
        row[f"{m} int"] = 100 * after[m][r].item()
    rows.append(row)
mean_row = {"rule": "Mean"}
for m in METRIC_NAMES:
    mean_row[f"{m} base"] = 100 * torch.nanmean(before[m]).item()
    mean_row[f"{m} int"] = 100 * torch.nanmean(after[m]).item()
rows.append(mean_row)
print(pd.DataFrame(rows).round(2).to_string(index=False))

## Empty Seat


In [ ]:
r = RULE_NAMES.index("Empty Seat")
logits = outputs[("coco_categories", "sum")]["logits"]
label = test_labels[:, r].bool()
before_ok = (torch.sigmoid(logits["none"][:, r]) > 0.5) == label
after_ok = (torch.sigmoid(logits["boost+suppress"][:, r]) > 0.5) == label
nb_mask = boundary_type[:, r] != 0
broken = nb_mask & before_ok & ~after_ok

def has(cat):
    return gt_test[:, category_id_by_name[cat]] > 0

n_nb = int(nb_mask.sum())
rows = []
for name, mask in [
    ("NB images", nb_mask),
    ("broken (correct before, wrong after)", broken),
    ("broken with person", broken & has("person")),
    ("broken with chair or couch", broken & (has("chair") | has("couch"))),
    ("broken with person and (chair or couch)", broken & has("person") & (has("chair") | has("couch"))),
]:
    rows.append({"": name, "count": int(mask.sum()), "% of NB images": 100 * int(mask.sum()) / n_nb})
print(pd.DataFrame(rows).round(2).to_string(index=False))